<div dir="rtl" style="text-align: right;">

# Function Calling — Phase 1
## Function Calling Fundamentals

**هدف فاز:** درک دقیق مفهوم Function Calling، نقش آن در معماری یک برنامه مبتنی بر LLM و شناخت چرخه‌ای که طی آن مدل درخواست اجرای یک ابزار را مطرح می‌کند و برنامه نتیجه را به مدل برمی‌گرداند.

در این فاز تمرکز روی فهم پایه و اجرای یک نمونه کوچک است. در فازهای بعدی طراحی Schema، اجرای چند ابزار، مدیریت خطا، امنیت و معماری Production را عمیق‌تر بررسی می‌کنیم.

### پیش‌نیازها
- آشنایی مقدماتی با Python و توابع
- آشنایی مقدماتی با LLM و پیام‌های `system`، `user` و `assistant`
- نصب و اجرای Ollama و در اختیار داشتن مدلی که از Tool Calling پشتیبانی کند

> نکته: نمونه‌های اجرایی این جزوه بر اساس مستندات رسمی Ollama و Python SDK آن نوشته شده‌اند. پشتیبانی از Tool Calling به مدل انتخاب‌شده وابسته است؛ اگر مدل شما این قابلیت را پشتیبانی نکند، صرفاً نصب SDK کافی نخواهد بود.

</div>

<div dir="rtl" style="text-align: right;">

## فهرست مطالب

1. What is Function Calling?
2. Function Calling vs Traditional Function Execution
3. Function Calling vs Structured Outputs
4. How LLMs Decide to Call Functions
5. Tools and Tool Calling Concepts
6. The Function Calling Lifecycle
7. Understanding Tool Calls and Tool Responses
8. LLM vs Application Responsibilities
9. نمونه عملی یکپارچه: بررسی مانده مرخصی
10. جمع‌بندی مفهومی و منابع رسمی

</div>

<div dir="rtl" style="text-align: right;">

## 1. What is Function Calling?

### تعریف

**Function Calling** یا **Tool Calling** روشی است که به کمک آن یک برنامه، قابلیت‌ها و ابزارهای مشخصی را در اختیار LLM قرار می‌دهد تا مدل بتواند در صورت نیاز درخواست استفاده از آن‌ها را تولید کند.

برای مثال، فرض کن کاربر از یک دستیار منابع انسانی می‌پرسد:

> How many vacation days do I have left?

مدل به‌تنهایی به دیتابیس منابع انسانی شرکت دسترسی ندارد. اما برنامه می‌تواند ابزاری با نام `get_leave_balance` در اختیار مدل قرار دهد. این ابزار وظیفه دارد مانده مرخصی یک کارمند را از یک منبع داده بخواند.

مدل ممکن است تشخیص دهد که برای پاسخ‌دادن باید این ابزار اجرا شود و در نتیجه یک Tool Call با نام تابع و آرگومان‌های لازم تولید کند.

**نکته بسیار مهم:** در الگوی معمول Function Calling، مدل خودش تابع Python را اجرا نمی‌کند و مستقیماً به دیتابیس یا API دسترسی پیدا نمی‌کند. مدل فقط درخواست ساختاریافته‌ای برای اجرای ابزار تولید می‌کند. این برنامه میزبان است که درخواست را دریافت، اعتبارسنجی و اجرا می‌کند.

### چرا به Function Calling نیاز داریم؟

LLM به‌صورت معمول بر اساس ورودی و دانشی که در اختیارش قرار گرفته پاسخ تولید می‌کند. اما بسیاری از کاربردهای واقعی به عملیاتی نیاز دارند که خارج از توانایی ذاتی مدل هستند، مانند:

- خواندن اطلاعات لحظه‌ای از دیتابیس
- فراخوانی یک REST API
- محاسبه با یک تابع قطعی در Python
- ثبت یک درخواست در سیستم سازمانی
- جست‌وجو در سامانه‌های داخلی
- ارسال یک عملیات به سرویس دیگری

Function Calling راهی استاندارد برای اتصال تصمیم‌گیری زبانی مدل به قابلیت‌های نرم‌افزار میزبان فراهم می‌کند.

### یک مثال ساده

بدون ابزار:

```text
User: How many vacation days do I have left?
LLM: I don't have access to your HR account, so I cannot know your balance.
```

با ابزار:

```text
User: How many vacation days do I have left?
LLM: Requests get_leave_balance(employee_id="E-104")
Application: Executes the function and gets 12.
LLM: You have 12 vacation days remaining.
```

در مثال دوم، عدد ۱۲ از نتیجه اجرای ابزار آمده است، نه از حافظه یا حدس مدل.

</div>

<div dir="rtl" style="text-align: right;">

## 2. Function Calling vs Traditional Function Execution

برای درک تفاوت، ابتدا یک تابع معمولی Python را در نظر بگیر:

```python
def get_leave_balance(employee_id: str) -> int:
    return 12

balance = get_leave_balance("E-104")
print(balance)
```

در این حالت، برنامه‌نویس مستقیماً تابع را فراخوانی کرده است. زمان و محل اجرای تابع از قبل در کد مشخص شده‌اند.

اما در Function Calling، برنامه تابع را به‌عنوان یک ابزار قابل استفاده به مدل معرفی می‌کند. سپس مدل بر اساس درخواست کاربر می‌تواند درخواست استفاده از آن ابزار را تولید کند.

### مقایسه

| ویژگی | Traditional Function Call | LLM Function Calling |
|---|---|---|
| تصمیم‌گیرنده برای فراخوانی | کد برنامه‌نویس | مدل، بر اساس ورودی و ابزارهای در دسترس |
| زمان فراخوانی | از قبل در منطق کد مشخص است | هنگام پردازش درخواست تعیین می‌شود |
| آرگومان‌ها | معمولاً توسط کد تعیین می‌شوند | مدل آن‌ها را از متن استخراج می‌کند |
| اجرای تابع | مستقیماً توسط برنامه | همچنان توسط برنامه میزبان |
| خروجی | مقدار بازگشتی تابع | نتیجه‌ای که برنامه به مدل برمی‌گرداند |

### نکته معماری

Function Calling جایگزین توابع معمولی Python نیست. در واقع، از توابع و سرویس‌های معمولی برنامه استفاده می‌کنیم؛ تفاوت در این است که LLM می‌تواند درخواست استفاده از آن‌ها را تولید کند.

بنابراین:

- تابع Python منطق واقعی عملیات را انجام می‌دهد.
- LLM درخواست کاربر را تفسیر و ابزار مناسب را پیشنهاد می‌کند.
- Application میان این دو هماهنگی ایجاد می‌کند.

</div>

<div dir="rtl" style="text-align: right;">

## 3. Function Calling vs Structured Outputs

این دو مفهوم به هم مرتبط‌اند، اما یک هدف ندارند.

### Structured Outputs چیست؟

گاهی می‌خواهیم مدل به‌جای تولید متن آزاد، خروجی را مطابق یک ساختار مشخص برگرداند؛ برای مثال یک JSON با فیلدهای `intent` و `employee_id`.

نمونه:

```json
{
  "intent": "check_leave_balance",
  "employee_id": "E-104"
}
```

در این حالت، هدف اصلی کنترل **ساختار خروجی مدل** است.

### Function Calling چه تفاوتی دارد؟

در Function Calling، مدل از میان ابزارهایی که در اختیارش قرار گرفته‌اند، ممکن است یکی را انتخاب کند و درخواست اجرای آن را همراه آرگومان‌ها تولید کند.

پس تفاوت اصلی:

- **Structured Outputs:** خروجی مدل باید چه ساختاری داشته باشد؟
- **Function Calling:** مدل برای استفاده از یک قابلیت خارجی، چه درخواست ابزاری تولید کند؟

### مثال مقایسه‌ای

فرض کن کاربر می‌گوید:

> Check my leave balance.

با Structured Output ممکن است مدل این JSON را تولید کند:

```json
{
  "action": "check_balance",
  "employee_id": "E-104"
}
```

این JSON به‌خودی‌خود باعث اجرای هیچ تابعی نمی‌شود. برنامه باید آن را بخواند و تصمیم بگیرد چه کاری انجام دهد.

با Function Calling، مدل یک Tool Call مشخص برای ابزار معرفی‌شده تولید می‌کند. برنامه آن Tool Call را دریافت می‌کند و در صورت مجاز بودن آن را اجرا می‌کند.

**نتیجه:** Structured Output برای شکل‌دهی خروجی است؛ Function Calling برای ایجاد یک قرارداد ساختاریافته جهت درخواست استفاده از ابزار. این دو می‌توانند در یک سیستم کنار هم استفاده شوند.

</div>

<div dir="rtl" style="text-align: right;">

## 4. How LLMs Decide to Call Functions

مدل معمولاً بر اساس سه دسته اطلاعات، پاسخ می‌سازد یا درخواست Tool Call تولید می‌کند:

1. **پیام کاربر:** کاربر چه چیزی می‌خواهد؟
2. **تعریف ابزارها:** چه ابزارهایی در اختیار مدل قرار گرفته‌اند و هرکدام چه کاری انجام می‌دهند؟
3. **دستورالعمل‌ها و Context:** مدل باید چه محدودیت‌ها و قواعدی را رعایت کند؟

### مثال

ابزار زیر را در اختیار مدل قرار داده‌ایم:

- Name: `get_leave_balance`
- Description: `Get the remaining annual leave days for an employee.`
- Parameter: `employee_id`

حالا دو درخواست را مقایسه کن:

**درخواست اول:**

> What is Function Calling?

برای پاسخ به این سؤال، احتمالاً نیازی به ابزار منابع انسانی نیست؛ مدل می‌تواند مستقیماً پاسخ متنی تولید کند.

**درخواست دوم:**

> Check the remaining annual leave days for employee E-104.

این درخواست به اطلاعات یک سامانه خارجی نیاز دارد؛ بنابراین ابزار `get_leave_balance` گزینه مناسبی است.

### آیا انتخاب ابزار همیشه درست است؟

خیر. مدل ممکن است:
- ابزار نامناسبی را انتخاب کند.
- مقدار یک پارامتر را اشتباه استخراج کند.
- با وجود نیاز نداشتن به ابزار، آن را درخواست کند.
- بدون ابزار پاسخ بدهد، درحالی‌که باید از داده خارجی استفاده کند.

به همین دلیل در فازهای بعدی درباره Schema Design، Validation، Tool Choice، Error Handling و Security صحبت خواهیم کرد.

**اصل مهم:** خروجی مدل یک درخواست برای انجام عملیات است، نه مجوز قطعی برای اجرای آن.

</div>

<div dir="rtl" style="text-align: right;">

## 5. Tools and Tool Calling Concepts

در Function Calling چند اصطلاح پایه وجود دارد که باید از همین ابتدا دقیق بشناسی.

### Tool

یک قابلیت یا عملیات قابل استفاده توسط مدل است؛ مانند:
- `get_leave_balance`
- `create_leave_request`
- `get_employee_profile`
- `search_company_policies`

Tool مفهومی گسترده‌تر از یک تابع ساده است. در عمل می‌تواند به یک تابع Python، یک سرویس Backend یا یک API خارجی متصل باشد.

### Function

تابع واقعی در کد برنامه است که عملیات را انجام می‌دهد.

مثلاً:

```python
def get_leave_balance(employee_id: str) -> int:
    ...
```

### Tool Definition

قراردادی است که به مدل معرفی می‌کند:
- ابزار چه نامی دارد؟
- چه کاری انجام می‌دهد؟
- چه پارامترهایی دریافت می‌کند؟
- ساختار پارامترها چیست؟

### Arguments

مقادیر ورودی‌ای هستند که مدل برای اجرای ابزار پیشنهاد می‌دهد.

مثلاً:

```json
{
  "employee_id": "E-104"
}
```

### Tool Call

درخواستی ساختاریافته است که مدل برای استفاده از یک ابزار تولید می‌کند. این درخواست معمولاً شامل نام ابزار و آرگومان‌ها است.

### Tool Result / Tool Response

نتیجه‌ای است که پس از اجرای ابزار توسط برنامه تولید می‌شود و دوباره در قالب پیام مناسب به مدل داده می‌شود.

### Assistant Message

پیامی است که از طرف مدل تولید می‌شود. این پیام ممکن است متن عادی داشته باشد، Tool Call داشته باشد یا بسته به API و پاسخ مدل شامل هر دو باشد.

### User Message

درخواست اولیه یا پیام بعدی کاربر است.

### Application / Host

برنامه‌ای است که مدل را فراخوانی می‌کند، ابزارها را معرفی می‌کند، Tool Callها را دریافت می‌کند، اجرای ابزارها را کنترل می‌کند و نتیجه را دوباره به مدل می‌دهد.

در این جزوه، واژه‌های Tool و Function را در بعضی توضیحات نزدیک به هم به کار می‌بریم؛ اما از نظر معماری، Tool قرارداد قابل مشاهده برای مدل است و Function می‌تواند پیاده‌سازی واقعی آن در برنامه باشد.

</div>

<div dir="rtl" style="text-align: right;">

## 6. The Function Calling Lifecycle

چرخه پایه Function Calling را می‌توان در چند مرحله دید.

### مرحله 1 — دریافت درخواست کاربر

کاربر یک درخواست به Application ارسال می‌کند.

```text
User:
How many vacation days do I have left?
```

### مرحله 2 — ارسال درخواست و ابزارها به LLM

Application پیام کاربر و تعریف ابزارهای در دسترس را برای مدل ارسال می‌کند.

در این مرحله تابع هنوز اجرا نشده است.

### مرحله 3 — تولید Tool Call توسط مدل

اگر مدل تشخیص دهد که ابزار لازم است، یک درخواست ساختاریافته تولید می‌کند:

```json
{
  "name": "get_leave_balance",
  "arguments": {
    "employee_id": "E-104"
  }
}
```

این صرفاً نمونه مفهومی است؛ ساختار دقیق پاسخ به API و SDK مورد استفاده وابسته است.

### مرحله 4 — دریافت و بررسی Tool Call

Application پاسخ مدل را بررسی می‌کند:
- آیا Tool Call وجود دارد؟
- نام ابزار چیست؟
- آرگومان‌ها چه هستند؟
- آیا ابزار در Registry برنامه وجود دارد؟
- آیا کاربر اجازه اجرای آن عملیات را دارد؟

### مرحله 5 — اجرای ابزار

Application تابع متناظر را اجرا می‌کند و نتیجه را دریافت می‌کند.

```python
result = get_leave_balance("E-104")
```

### مرحله 6 — بازگرداندن نتیجه به مدل

Application نتیجه ابزار را در قالب پیام Tool به مکالمه اضافه می‌کند و درخواست دیگری به مدل می‌فرستد.

```text
Tool result:
12
```

### مرحله 7 — تولید پاسخ نهایی

مدل از نتیجه ابزار برای ساخت پاسخ قابل فهم برای کاربر استفاده می‌کند.

```text
You have 12 vacation days remaining.
```

### دیاگرام چرخه

```text
User
  |
  v
Application
  |
  | user message + tool definitions
  v
LLM
  |
  | tool call request
  v
Application
  |
  | validate + execute
  v
Tool / Function
  |
  | tool result
  v
Application
  |
  | tool result + conversation history
  v
LLM
  |
  | final response
  v
User
```

### نکته کلیدی

در این چرخه معمولاً بیش از یک درخواست به LLM وجود دارد:
- درخواست اول برای تحلیل پیام و احتمالاً تولید Tool Call
- درخواست بعدی برای استفاده از نتیجه ابزار و تولید پاسخ نهایی

تعداد دقیق درخواست‌ها به سناریو، مدل و طراحی برنامه بستگی دارد.

</div>

<div dir="rtl" style="text-align: right;">

## 7. Understanding Tool Calls and Tool Responses

یکی از اشتباهات رایج در شروع یادگیری این است که Tool Call را با نتیجه اجرای تابع اشتباه بگیریم.

این دو مرحله کاملاً متفاوت‌اند.

### Tool Call

مدل می‌گوید:

> لطفاً ابزار `get_leave_balance` را با آرگومان `employee_id="E-104"` اجرا کن.

این پیام هنوز نتیجه‌ای از دیتابیس ندارد.

### Tool Execution

برنامه درخواست مدل را دریافت می‌کند، تابع را اجرا می‌کند و مثلاً عدد ۱۲ را از دیتابیس می‌خواند.

### Tool Response

برنامه نتیجه را در قالب مورد انتظار API به مکالمه برمی‌گرداند.

### Final Assistant Response

مدل نتیجه را دریافت می‌کند و آن را به شکل پاسخ طبیعی به کاربر ارائه می‌دهد.

| مرحله | تولیدکننده | مفهوم |
|---|---|---|
| Tool Call | LLM | درخواست استفاده از ابزار |
| Tool Execution | Application | اجرای واقعی عملیات |
| Tool Result | Application / Tool | نتیجه اجرای عملیات |
| Final Response | LLM | پاسخ نهایی به کاربر |

### چرا این تفکیک مهم است؟

چون ممکن است:
- Tool Call معتبر باشد اما اجرای تابع شکست بخورد.
- تابع اجرا شود اما نتیجه خالی باشد.
- نتیجه تابع نیازمند تبدیل یا خلاصه‌سازی باشد.
- کاربر مجوز اجرای عملیات را نداشته باشد.

بنابراین نباید صرفاً با دیدن یک Tool Call فرض کنیم عملیات انجام شده است.

</div>

<div dir="rtl" style="text-align: right;">

## 8. LLM vs Application Responsibilities

در یک معماری صحیح باید مسئولیت مدل و برنامه را از هم جدا کنیم.

### مسئولیت‌های LLM

- فهم درخواست زبانی کاربر
- تشخیص احتمالی نیاز به ابزار
- انتخاب ابزار از میان ابزارهای معرفی‌شده
- پیشنهاد آرگومان‌های ابزار
- استفاده از نتیجه ابزار برای ساخت پاسخ نهایی

### مسئولیت‌های Application

- نگهداری و ارسال تاریخچه مکالمه
- معرفی ابزارها به مدل
- دریافت و تفسیر Tool Call
- بررسی نام ابزار و آرگومان‌ها
- اعتبارسنجی ورودی‌ها
- کنترل مجوزها و سیاست‌های دسترسی
- اجرای واقعی تابع یا سرویس
- مدیریت خطا و محدودیت‌های اجرا
- بازگرداندن نتیجه ابزار به مدل
- ثبت Log و کنترل رفتار سیستم

### چرا نباید اجرای عملیات را به تصمیم مدل واگذار کنیم؟

مدل یک مؤلفه احتمالاتی است و ممکن است درخواست نادرست تولید کند. بنابراین برای عملیات حساس، Application باید مستقلاً بررسی کند که:
- کاربر احراز هویت شده است؟
- اجازه دسترسی به این داده را دارد؟
- آرگومان‌ها معتبرند؟
- عملیات در وضعیت فعلی مجاز است؟
- آیا تأیید انسانی لازم است؟

برای مثال، اگر کاربر درخواست ثبت مرخصی داشته باشد، مدل می‌تواند Tool Call مربوط به ثبت درخواست را پیشنهاد دهد؛ اما Backend باید مالک اعتبارسنجی قوانین مرخصی و مجوز ثبت باشد.

**قاعده معماری:** LLM تصمیم پیشنهادی را تولید می‌کند؛ Application و Backend اجرای کنترل‌شده را بر عهده دارند.

</div>

<div dir="rtl" style="text-align: right;">

# 9. Practical Example
## بررسی مانده مرخصی با Ollama و Python

در این مثال یک سناریوی کوچک HR را پیاده‌سازی می‌کنیم.

کاربر می‌پرسد:

> How many annual leave days do I have left?

مدل به ابزار `get_leave_balance` دسترسی دارد. اگر ابزار را درخواست کند، برنامه آن را اجرا می‌کند و نتیجه را دوباره به مدل برمی‌گرداند.

### 9.1 نصب وابستگی

برای این نمونه از Python SDK رسمی Ollama استفاده می‌کنیم.

در Terminal یا یک سلول Notebook اجرا کن:

</div>

In [ ]:
%pip install -U ollama

<div dir="rtl" style="text-align: right;">

### 9.2 بررسی اتصال و انتخاب مدل

ابتدا مطمئن شو Ollama در حال اجراست و مدلی که انتخاب می‌کنی از Tool Calling پشتیبانی می‌کند.

در مستندات رسمی Ollama نمونه‌هایی با مدل `qwen3` ارائه شده‌اند. اگر مدل دیگری استفاده می‌کنی، نام آن را در متغیر `MODEL_NAME` قرار بده و پشتیبانی Tool Calling آن را بررسی کن.

</div>

In [11]:
from ollama import chat

MODEL_NAME = "llama3.2"

response = chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ]
)

print(response.message.content)

Hello!


<div dir="rtl" style="text-align: right;">

### 9.3 تعریف تابع واقعی

این تابع در این مثال به‌جای دیتابیس، از یک Dictionary استفاده می‌کند تا تمرکز ما روی چرخه Function Calling باشد.

در پروژه واقعی، پیاده‌سازی این تابع می‌تواند از Repository، دیتابیس یا یک سرویس HR استفاده کند.

</div>

In [12]:
employees = {
    "E-104": {
        "name": "Alex",
        "annual_leave_remaining": 12
    },
    "E-205": {
        "name": "Sam",
        "annual_leave_remaining": 7
    }
}


def get_leave_balance(employee_id: str) -> dict:
    employee = employees.get(employee_id)

    if employee is None:
        return {
            "success": False,
            "error": "Employee not found."
        }

    return {
        "success": True,
        "employee_id": employee_id,
        "employee_name": employee["name"],
        "remaining_days": employee["annual_leave_remaining"]
    }

<div dir="rtl" style="text-align: right;">

### 9.4 معرفی Tool به مدل

در این مرحله تابع را به‌صورت یک Tool Definition در اختیار مدل قرار می‌دهیم.

به سه قسمت دقت کن:

- `name`: نام ابزار
- `description`: توضیح کاری که ابزار انجام می‌دهد
- `parameters`: قرارداد ورودی ابزار بر اساس JSON Schema

فعلاً فقط یک پارامتر اجباری داریم: `employee_id`.

</div>

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_leave_balance",
            "description": (
                "Get the remaining annual leave days for an employee. "
                "Use this tool when the user asks about an employee's "
                "remaining annual leave balance."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string",
                        "description": "The unique employee identifier."
                    }
                },
                "required": ["employee_id"]
            }
        }
    }
]

<div dir="rtl" style="text-align: right;">

### 9.5 ارسال درخواست کاربر به مدل

اکنون پیام کاربر و ابزار را برای مدل ارسال می‌کنیم.

درخواست کاربر شامل شناسه کارمند است تا مدل بتواند آن را به آرگومان ابزار تبدیل کند.

</div>

In [16]:
messages = [
    {
        "role": "user",
        "content": (
            "How many annual leave days does employee E-104 have left?"
        )
    }
]

response = chat(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

print("Assistant message:")
print(response.message)

Assistant message:
role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='get_leave_balance', arguments={'employee_id': 'E-104'}))]


<div dir="rtl" style="text-align: right;">

### 9.6 بررسی Tool Call

پاسخ مدل را بررسی می‌کنیم تا ببینیم آیا Tool Call تولید شده است یا خیر.

در Ollama Python SDK، Tool Callها در `response.message.tool_calls` قابل مشاهده‌اند.

نکته: نام ابزار و آرگومان‌ها را از پاسخ واقعی مدل بخوان؛ خروجی مدل را از قبل قطعی فرض نکن.

</div>

In [17]:
tool_calls = response.message.tool_calls or []

print("Number of tool calls:", len(tool_calls))

for call in tool_calls:
    print("Tool name:", call.function.name)
    print("Arguments:", call.function.arguments)

Number of tool calls: 1
Tool name: get_leave_balance
Arguments: {'employee_id': 'E-104'}


<div dir="rtl" style="text-align: right;">

### 9.7 اجرای ابزار در Application

حالا Application باید نام ابزار را بررسی کند و تابع متناظر را اجرا کند.

در این نمونه برای سادگی یک ابزار داریم. در فازهای بعدی یک Tool Registry عمومی‌تر طراحی می‌کنیم.

</div>

In [18]:
tool_results = []

for call in tool_calls:
    if call.function.name == "get_leave_balance":
        arguments = call.function.arguments

        result = get_leave_balance(
            employee_id=arguments["employee_id"]
        )
    else:
        result = {
            "success": False,
            "error": f"Unknown tool: {call.function.name}"
        }

    tool_results.append({
        "tool_name": call.function.name,
        "result": result
    })

print(tool_results)

[{'tool_name': 'get_leave_balance', 'result': {'success': True, 'employee_id': 'E-104', 'employee_name': 'Alex', 'remaining_days': 12}}]


<div dir="rtl" style="text-align: right;">

### 9.8 بازگرداندن نتیجه ابزار به مدل

برای اینکه مدل بتواند پاسخ نهایی تولید کند، باید پیام Assistant حاوی Tool Call و سپس نتیجه ابزار را به تاریخچه مکالمه اضافه کنیم.

در SDK Ollama می‌توانیم خود پیام Assistant را به تاریخچه اضافه کنیم و سپس پاسخ ابزار را با نقش `tool` برگردانیم.

</div>

In [19]:
messages.append(response.message)

for item in tool_results:
    messages.append({
        "role": "tool",
        "tool_name": item["tool_name"],
        "content": str(item["result"])
    })

print(messages)

[{'role': 'user', 'content': 'How many annual leave days does employee E-104 have left?'}, Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_leave_balance', arguments={'employee_id': 'E-104'}))]), {'role': 'tool', 'tool_name': 'get_leave_balance', 'content': "{'success': True, 'employee_id': 'E-104', 'employee_name': 'Alex', 'remaining_days': 12}"}]


<div dir="rtl" style="text-align: right;">

### 9.9 دریافت پاسخ نهایی

درخواست دوم را ارسال می‌کنیم. این بار مدل علاوه بر پیام کاربر و Tool Call قبلی، نتیجه واقعی ابزار را هم در تاریخچه دارد.

</div>

In [20]:
final_response = chat(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

print(final_response.message.content)

Employee E-104 has 12 annual leave days left.


<div dir="rtl" style="text-align: right;">

### 9.10 نسخه یکپارچه چرخه

کد زیر همان مراحل را در یک جریان پیوسته نشان می‌دهد. این نسخه برای درک ارتباط اجزای اصلی مفید است؛ هنوز یک پیاده‌سازی Production-ready نیست و در فازهای بعدی محدودیت‌ها و کنترل‌های لازم را اضافه می‌کنیم.

</div>

In [21]:
from ollama import chat

MODEL_NAME = "llama3.2"

employees = {
    "E-104": {"name": "Alex", "annual_leave_remaining": 12},
    "E-205": {"name": "Sam", "annual_leave_remaining": 7}
}


def get_leave_balance(employee_id: str) -> dict:
    employee = employees.get(employee_id)

    if employee is None:
        return {"success": False, "error": "Employee not found."}

    return {
        "success": True,
        "employee_id": employee_id,
        "employee_name": employee["name"],
        "remaining_days": employee["annual_leave_remaining"]
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_leave_balance",
            "description": "Get the remaining annual leave days for an employee.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string",
                        "description": "The unique employee identifier."
                    }
                },
                "required": ["employee_id"]
            }
        }
    }
]

messages = [
    {
        "role": "user",
        "content": "How many annual leave days does employee E-104 have left?"
    }
]

# 1. Send the user message and available tools to the model.
response = chat(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

# 2. Preserve the assistant message in the conversation.
messages.append(response.message)

# 3. Execute every requested tool call in the host application.
for call in response.message.tool_calls or []:
    if call.function.name == "get_leave_balance":
        result = get_leave_balance(
            employee_id=call.function.arguments["employee_id"]
        )
    else:
        result = {"success": False, "error": "Unknown tool."}

    # 4. Return the tool result to the conversation.
    messages.append({
        "role": "tool",
        "tool_name": call.function.name,
        "content": str(result)
    })

# 5. Ask the model to produce a user-facing answer.
final_response = chat(
    model=MODEL_NAME,
    messages=messages,
    tools=tools
)

print(final_response.message.content)

Employee E-104 has 12 annual leave days left.


<div dir="rtl" style="text-align: right;">

### 9.11 چه اتفاقی در این مثال افتاد؟

| مرحله | اتفاق |
|---|---|
| 1 | پیام کاربر ساخته شد. |
| 2 | Tool Definition به مدل معرفی شد. |
| 3 | مدل درخواست Tool Call تولید کرد. |
| 4 | Application نام ابزار و آرگومان را خواند. |
| 5 | تابع Python اجرا شد. |
| 6 | نتیجه تابع به تاریخچه مکالمه اضافه شد. |
| 7 | مدل با استفاده از نتیجه ابزار پاسخ نهایی تولید کرد. |

### محدودیت‌های نسخه آموزشی

این کد برای فهم چرخه پایه طراحی شده و برای استفاده عملیاتی کافی نیست. برای نمونه:
- فرض کرده‌ایم مدل Tool Call مورد انتظار را تولید می‌کند.
- اعتبارسنجی کامل آرگومان‌ها را پیاده‌سازی نکرده‌ایم.
- احراز هویت و مجوز دسترسی کاربر وجود ندارد.
- مدیریت خطاهای شبکه و اجرای ابزار کامل نیست.
- سقف تعداد مراحل و جلوگیری از حلقه بی‌نهایت را اضافه نکرده‌ایم.
- برای داده واقعی باید از منبع داده و Backend امن استفاده شود.

این موارد موضوع فازهای بعدی هستند.

</div>

<div dir="rtl" style="text-align: right;">

# 10. جمع‌بندی مفهومی و منابع رسمی

## مفاهیمی که باید از این فاز به خاطر داشته باشی

1. Function Calling راهی برای متصل‌کردن LLM به ابزارها و قابلیت‌های نرم‌افزار است.
2. مدل معمولاً تابع را مستقیماً اجرا نمی‌کند؛ Tool Call را به Application تحویل می‌دهد.
3. Application مسئول بررسی و اجرای واقعی ابزار است.
4. Tool Definition قرارداد ابزار را به مدل معرفی می‌کند.
5. Tool Call با Tool Result متفاوت است.
6. پس از اجرای ابزار، نتیجه باید به مدل برگردانده شود تا مدل بتواند پاسخ نهایی را تولید کند.
7. Structured Outputs و Function Calling مفاهیم مرتبط اما با اهداف متفاوت‌اند.
8. خروجی مدل نباید به‌عنوان مجوز اجرای عملیات حساس تلقی شود.

## منابع رسمی برای مطالعه بیشتر

- [Ollama — Tool Calling](https://docs.ollama.com/capabilities/tool-calling)
- [Ollama — Python Library](https://github.com/ollama/ollama-python)
- [OpenAI API — Function Calling](https://developers.openai.com/api/docs/guides/function-calling)
- [OpenAI API — Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)

> مستندات Providerها ممکن است در جزئیات Schema، ساختار پاسخ و نحوه ارسال Tool Result متفاوت باشند. در این جزوه، کد عملی بر اساس Ollama نوشته شده و مفاهیم معماری به‌صورت عمومی توضیح داده شده‌اند.

</div>